# SNOM Connection & Movement Test
Tests `nea_tools` connection, motor read-back, and XY movement via `NeaSnomBackend`.

## 0 — Configuration

In [ ]:
HOST        = 'nea-server'   # change to IP if DNS does not resolve
PATH_TO_DLL = ''             # leave empty unless nea_tools asks for a specific path

# Movement test: the stage will visit these four corners (nm)
TEST_POSITIONS = [
    (    0.0,     0.0),   # origin
    ( 1000.0,     0.0),   # +1 µm X
    ( 1000.0,  1000.0),   # +1 µm X, +1 µm Y
    (    0.0,  1000.0),   # +1 µm Y
    (    0.0,     0.0),   # back to origin
]

## 1 — Imports

In [ ]:
import sys, time
import asyncio
import nest_asyncio
import nea_tools
print('nea_tools version:', getattr(nea_tools, '__version__', 'unknown'))

## 2 — Connect to nea-server

In [ ]:
loop = asyncio.get_event_loop()
nest_asyncio.apply(loop)

print(f'Connecting to {HOST} ...')
loop.run_until_complete(nea_tools.connect(HOST, fingerprint=None, path_to_dll=PATH_TO_DLL))
print('Connected.')

## 3 — Import neaspec (injected after connect)

In [ ]:
from neaspec import context
from nea_tools.microscope import motors
print('neaspec context:', context)
print('motors module  :', motors)

## 4 — Read current position

In [ ]:
pos = loop.run_until_complete(
    context.Microscope.GetActiveMotorDistanceToReferenceXyz()
)
print(f'Current position  X={pos.X:.1f} nm   Y={pos.Y:.1f} nm   Z={pos.Z:.1f} nm')

## 5 — Read SNOM signals (snapshot)

In [ ]:
mic = context.Microscope.Py

print('Harmonic  |  O_Amp      |  M_Amp')
print('-' * 40)
for h in range(6):
    try:
        oa = float(mic.OpticalAmplitude(h))
    except Exception as e:
        oa = float('nan')
        print(f'  O_Amp({h}) error: {e}')
    try:
        ma = float(mic.MechanicalAmplitude(h))
    except Exception as e:
        ma = float('nan')
        print(f'  M_Amp({h}) error: {e}')
    print(f'  {h}       |  {oa:10.4f}  |  {ma:10.4f}')

## 6 — Movement test
Visits each position in `TEST_POSITIONS`, reads back XY, and prints the error.

In [ ]:
results = []

for (x_target, y_target) in TEST_POSITIONS:
    print(f'Moving to ({x_target:.0f}, {y_target:.0f}) nm ...', end=' ')
    t0 = time.time()

    with motors.Sample() as sample:
        sample.activate()
        ok = loop.run_until_complete(
            sample.ActiveMotorGotoXyzAsync((x_target, y_target, 0.0))
        )

    dt = time.time() - t0

    pos = loop.run_until_complete(
        context.Microscope.GetActiveMotorDistanceToReferenceXyz()
    )
    err_x = pos.X - x_target
    err_y = pos.Y - y_target

    status = 'OK' if ok else 'FAILED'
    print(f'{status}  |  readback ({pos.X:.1f}, {pos.Y:.1f}) nm  |  err ({err_x:+.1f}, {err_y:+.1f}) nm  |  {dt:.2f} s')
    results.append(dict(target=(x_target, y_target), readback=(pos.X, pos.Y),
                        err_x=err_x, err_y=err_y, ok=ok, dt_s=dt))

print('\nDone.')

## 7 — Summary table

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df['target_x'] = df['target'].apply(lambda t: t[0])
df['target_y'] = df['target'].apply(lambda t: t[1])
df['readback_x'] = df['readback'].apply(lambda r: r[0])
df['readback_y'] = df['readback'].apply(lambda r: r[1])
df[['target_x','target_y','readback_x','readback_y','err_x','err_y','ok','dt_s']]

## 8 — Disconnect

In [ ]:
try:
    nea_tools.disconnect()
    print('Disconnected.')
except Exception as e:
    print(f'Disconnect error (ignored): {e}')